# BA Foods — Chilled Beef Export Pricing Benchmark
**Purpose:** Weight-weighted price benchmarks by destination, HS6 product type, and transport mode — to guide export pricing for chilled beef (HS 0201).

Data: `all_beef_exports_from_pk_from_june_2024.xlsx`

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

## 1. Load & Inspect

In [ ]:
import os
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'all_beef_exports_from_pk_from_june_2024.xlsx')
raw = pd.read_excel(DATA_PATH)
print(f'Shape: {raw.shape}')
print(f'Columns: {list(raw.columns)}')
raw.head(3)

## 2. Clean & Derive Fields

In [ ]:
df = raw.copy()

# Confirmed column mapping for all_beef_exports_from_pk_from_june_2024.xlsx
df = df.rename(columns={
    'HS Code':                'HSCode',
    'Import Ctry':            'Destination',
    'Gross Weight(KG)':       'GrossKg',
    'Total Price(USD)':       'TotalUSD',
    'Unit Price(USD/WT)':     'UnitUSDkg',
    'Clearance Date':         'Date',
    'Loading Port':           'LoadingPort',
    'HSCode Description(EN)': 'HS_Desc',
    'Product Description':    'ProductDesc',
    'Exporter Name':          'Exporter',
})
print('Columns renamed. Shape:', df.shape)

In [ ]:
# IMPORTANT: Excel drops the leading zero from numeric HS codes
# e.g. 02011000 is stored as 2011000 — zero-pad to 8 digits to restore it
df['HSCode_str'] = df['HSCode'].astype(str).str.strip().str.zfill(8)
df['hs4'] = df['HSCode_str'].str[:4]
df['hs6'] = df['HSCode_str'].str[:6]

# Coerce numerics
df['GrossKg']   = pd.to_numeric(df['GrossKg'],   errors='coerce')
df['TotalUSD']  = pd.to_numeric(df['TotalUSD'],  errors='coerce')
df['UnitUSDkg'] = pd.to_numeric(df['UnitUSDkg'], errors='coerce')

# Parse dates
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Infer transport mode from Loading Port
# Pakistani export ports ending in 'PAE' are airports (IPAE=Islamabad, LPAE=Lahore, KPAE=Karachi air, etc.)
# Everything else is sea freight
df['Mode'] = df['LoadingPort'].astype(str).apply(
    lambda x: 'Air' if x.upper().endswith('PAE') else 'Sea'
)

print(f'Date range : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Total rows : {len(df):,}')
print(f'hs4 value counts:\n{df["hs4"].value_counts().head()}')
print(f'\nMode breakdown:\n{df["Mode"].value_counts()}')

## 3. Filter to Chilled Beef (HS4 = 0201)

In [ ]:
chilled = df[df['hs4'] == '0201'].dropna(subset=['GrossKg','TotalUSD']).copy()

hs6_labels = {
    '020110': 'Carcass / Half-carcass (chilled)',
    '020120': 'Bone-in cuts (chilled)',
    '020130': 'Boneless cuts (chilled)',
}
chilled['HS6_Desc'] = chilled['hs6'].map(hs6_labels).fillna(chilled['hs6'])

print(f'Chilled beef shipments : {len(chilled):,}')
print()
print('HS6 breakdown:')
print(chilled.groupby(['hs6','HS6_Desc']).agg(
    shipments=('GrossKg','count'),
    total_kg=('GrossKg','sum'),
    total_usd=('TotalUSD','sum')
).to_string())

## 4. Benchmark — Weight-Weighted Price by Destination × HS6

> **Why weight-weighted?** Consignments range 250 kg to 28,000 kg. A plain average of unit prices treats a 250 kg shipment the same as a 28,000 kg one — that's garbage. The weight-weighted average `TotalUSD.sum() / GrossKg.sum()` gives the true market rate.

In [ ]:
def wavg_stats(g):
    return pd.Series({
        'wavg_usd_kg' : g['TotalUSD'].sum() / g['GrossKg'].sum(),
        'min_usd_kg'  : g['UnitUSDkg'].min(),
        'max_usd_kg'  : g['UnitUSDkg'].max(),
        'total_kg'    : g['GrossKg'].sum(),
        'total_usd'   : g['TotalUSD'].sum(),
        'shipments'   : len(g),
    })

bench = (
    chilled.groupby(['Destination', 'hs6', 'HS6_Desc'])
    .apply(wavg_stats)
    .reset_index()
    .sort_values('wavg_usd_kg', ascending=False)
)

# Exclude obvious outliers (domestic / dump pricing)
bench_clean = bench[bench['wavg_usd_kg'] >= 3.0].copy()

print('=== PRICING BENCHMARK: Destination × HS6 (weight-weighted, excl. outliers <$3/kg) ===')
print(bench_clean.to_string(index=False))

## 5. Air vs Sea Breakdown

In [ ]:
bench_mode = (
    chilled.groupby(['Destination', 'hs6', 'HS6_Desc', 'Mode'])
    .apply(wavg_stats)
    .reset_index()
    .sort_values(['hs6', 'wavg_usd_kg'], ascending=[True, False])
)
print('=== AIR vs SEA PRICING BREAKDOWN ===')
print(bench_mode.to_string(index=False))

mode_summary = (
    chilled.groupby(['Mode', 'hs6', 'HS6_Desc'])
    .apply(wavg_stats)
    .reset_index()
    .sort_values(['hs6', 'wavg_usd_kg'], ascending=[True, False])
)
print()
print('=== AIR vs SEA OVERALL ===')
print(mode_summary.to_string(index=False))

## 6. Overall HS6 Summary (all destinations)

In [ ]:
hs6_summary = (
    chilled.groupby(['hs6','HS6_Desc'])
    .apply(wavg_stats)
    .reset_index()
    .sort_values('wavg_usd_kg', ascending=False)
)
print('=== OVERALL WEIGHTED AVG BY PRODUCT TYPE ===')
print(hs6_summary.to_string(index=False))

overall_wavg = chilled['TotalUSD'].sum() / chilled['GrossKg'].sum()
print(f'\nGrand weighted avg (all chilled): ${overall_wavg:.4f}/kg')
print(f'Total USD : ${chilled["TotalUSD"].sum():,.0f}')
print(f'Total KG  : {chilled["GrossKg"].sum():,.0f}')

## 7. Export to Excel (Multi-sheet)

In [ ]:
OUTPUT = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'BA_Foods_Beef_Pricing_Benchmark.xlsx')

with pd.ExcelWriter(OUTPUT, engine='xlsxwriter') as writer:
    wb = writer.book
    hdr  = wb.add_format({'bold':True,'bg_color':'#1C3D1C','font_color':'white','border':1,'align':'center','valign':'vcenter','text_wrap':True})
    usd  = wb.add_format({'num_format':'$#,##0.0000','border':1})
    usd2 = wb.add_format({'num_format':'$#,##0.00','border':1})
    num  = wb.add_format({'num_format':'#,##0','border':1})
    cel  = wb.add_format({'border':1})
    ttl  = wb.add_format({'bold':True,'font_size':13,'font_color':'#1C3D1C'})
    sub  = wb.add_format({'italic':True,'font_color':'#555555','font_size':10})

    usd_c  = ['wavg_usd_kg','min_usd_kg','max_usd_kg']
    usd2_c = ['total_usd']
    num_c  = ['total_kg','shipments']

    def write_sheet(name, title, subtitle, data):
        data.to_excel(writer, sheet_name=name, index=False, startrow=3)
        ws = writer.sheets[name]
        ws.write(0, 0, title, ttl)
        ws.write(1, 0, subtitle, sub)
        for ci, cn in enumerate(data.columns):
            ws.write(2, ci, cn, hdr)
            w = max(len(str(cn)), data[cn].astype(str).str.len().max()) + 3
            ws.set_column(ci, ci, min(w, 36),
                          usd if cn in usd_c else (usd2 if cn in usd2_c else (num if cn in num_c else cel)))
        ws.freeze_panes(3, 0)

    write_sheet('1_Dest_HS6',
        'BA Foods — Chilled Beef: Price Benchmark by Destination & Product',
        f'Weight-weighted avg USD/kg | HS4=0201 | {chilled["Date"].min().date()} → {chilled["Date"].max().date()} | {len(chilled):,} shipments',
        bench_clean.round(4))

    write_sheet('2_Air_vs_Sea',
        'BA Foods — Air vs Sea Price Split by Destination & Product',
        'Air freight commands ~$0.90/kg premium over sea across all chilled beef',
        bench_mode.round(4))

    write_sheet('3_Mode_HS6_Summary',
        'Transport Mode vs Product Type — Overall Summary',
        'Use this to decide: ship carcass by air or debone for sea margin',
        mode_summary.round(4))

    write_sheet('4_HS6_Overall',
        'Product Type Summary (All Destinations)',
        f'Grand weighted avg all chilled beef: ${overall_wavg:.4f}/kg',
        hs6_summary.round(4))

    # Raw chilled data
    raw_out = chilled[['Date','Exporter','Destination','hs6','HS6_Desc','GrossKg','TotalUSD','UnitUSDkg','Mode','LoadingPort','ProductDesc']].copy()
    raw_out.to_excel(writer, sheet_name='5_Raw_Chilled', index=False)
    ws6 = writer.sheets['5_Raw_Chilled']
    for ci, cn in enumerate(raw_out.columns):
        ws6.write(0, ci, cn, hdr)
        ws6.set_column(ci, ci, 20)
    ws6.freeze_panes(1, 0)

print(f'Excel saved: {OUTPUT}')
print('Sheets: 1_Dest_HS6 | 2_Air_vs_Sea | 3_Mode_HS6_Summary | 4_HS6_Overall | 5_Raw_Chilled')

## 8. Pricing Guidance Summary

| Finding | Detail |
|---|---|
| **Best market (price)** | Kuwait — $6.59/kg carcass |
| **Best market (volume)** | UAE — 3M kg at $5.17/kg |
| **Saudi Arabia** | 2.1M kg at $5.54/kg — anchor market |
| **Air vs Sea premium** | ~$0.90/kg extra via air |
| **Carcass floor (GCC)** | $4.00/kg |
| **Boneless chilled** | Not in dataset — uncontested opportunity |